# 🛍️ DORMIY RETAIL MULTIMODAL SYSTEM: DUA ALUR KERJA TERPADU
### Model Terpadu: CLIP ViT (Fine-Tuned Sesi 3) + SVM Augmented (250 Sampel) + Cosine Similarity Engine

---

## 📌 Dua Alur Kerja Utama Sistem (Dual-Workflow Architecture)

```mermaid
flowchart TD
    subgraph ALUR_1["🏢 ALUR 1: ADMIN MENAMBAHKAN PRODUK BARU (Ingestion & Auto-Categorization)"]
        A1["1. Admin Input: Foto Produk Baru + Nama Produk"] --> B1["2. CLIP Image Encoder Ekstrak Vektor 512 Dimensi"]
        B1 --> C1["3. Vektor Masuk ke SVM"]
        C1 --> D1["4. SVM Menetapkan Kategori Otomatis (Top-1 & Top-2 Confidence)"]
        B1 --> E1[("5. Vektor Disimpan ke Feature Store Database Katalog")]
    end

    subgraph ALUR_2["🛒 ALUR 2: USER MENCARI DENGAN TEKS BEBAS (Search & Top-5 Retrieval)"]
        A2["1. User Input Query Teks (Misal: 'stationery and writing tools')"] --> B2["2. CLIP Text Encoder Ekstrak Vektor 512 Dimensi"]
        B2 --> C2["3. SVM Mendeteksi Kategori Query -> 'Office Supplies'"]
        B2 --> D2["4. Cosine Similarity Menghitung Kemiripan ke Seluruh Vektor Database"]
        E1 -. Dibandingkan .-> D2
        D2 --> E2["5. Hasil di Layar: Kategori Terdeteksi + Top-5 Produk Paling Cocok"]
    end
```

---

## 💡 Rangkuman Performa Model:
* **Model Kategori (SVM Terkalibrasi + Data Augmentasi 250 Sampel)**:
  - **Top-1 Category Accuracy**: **`86.00%`**
  - **Top-2 Category Accuracy**: **`100.00%`** (Seluruh barang selalu tepat masuk di 2 kategori teratas!)
* **Model Produk Spesifik (CLIP Sesi 3 + Cosine Similarity)**:
  - **Top-1 Product Retrieval**: **`84.00%`**
  - **Top-5 Product Retrieval**: **`96.00%`**


--- 
## 1. Import Library & Inisialisasi Sistem

In [1]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
from dormiy_production_pipeline import DormiyRetailSystem

# Inisialisasi Sistem Retail Tunggal yang menyatukan kedua alur
system = DormiyRetailSystem(
    clip_weights_dir="experiment_results/sesi_3_projection_head/model_clip_dormiy_sesi3",
    svm_model_path="experiment_results/sesi_augmented_svm/svm_augmented_classifier.joblib",
    manifest_csv_path="DORMIY-ML-Dataset/data/final/cv/dormiy_cv_manifest.csv",
    device="cpu"  # atau 'cuda'
)

Menginisialisasi DORMIY Retail System pada device: CPU...
Loading CLIP weights from: experiment_results/sesi_3_projection_head/model_clip_dormiy_sesi3


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading catalog manifest from: DORMIY-ML-Dataset/data/final/cv/dormiy_cv_manifest.csv
Loading SVM Category Classifier from: experiment_results/sesi_augmented_svm/svm_augmented_classifier.joblib


Membangun Feature Store Embeddings...


✅ Sistem Siap! Total Produk di Katalog: 50 item.



--- 
## 2. Demonstrasi ALUR 1: Admin Memasukkan Gambar & Nama Produk Baru
Admin mengunggah foto dan judul barang. CLIP mengekstrak vektor visual, dan SVM otomatis mengelompokkan kategori yang cocok.

In [2]:
# Contoh: Admin memasukkan foto produk baru
sample_row = system.df_catalog.iloc[0]
test_image_path = sample_row['full_image_path']
test_product_title = "Gagang Kayu Pembersih Lantai Asrama 60 Inci"

result_alur1 = system.admin_add_new_product(
    image_input=test_image_path,
    product_title=test_product_title,
    save_to_catalog=False  # Set True jika ingin langsung disimpan permanen ke memori katalog
)

print("=" * 85)
print(f" 🏢 {result_alur1['workflow']}")
print(f" 📦 NAMA PRODUK DIINPUT : {result_alur1['product_title']}")
print(f" 🏷️ REKOMENDASI KATEGORI : {result_alur1['svm_category_prediction']['recommended_category']} (Keyakinan: {result_alur1['svm_category_prediction']['confidence_pct']}%)")
print(f" 🥈 KATEGORI CADANGAN    : {result_alur1['svm_category_prediction']['secondary_category']} (Keyakinan: {result_alur1['svm_category_prediction']['secondary_confidence_pct']}%)")
print("=" * 85)


 🏢 ALUR 1: Admin Product Ingestion
 📦 NAMA PRODUK DIINPUT : Gagang Kayu Pembersih Lantai Asrama 60 Inci
 🏷️ REKOMENDASI KATEGORI : Hardware (Keyakinan: 49.46%)
 🥈 KATEGORI CADANGAN    : Home & Garden (Keyakinan: 25.73%)


--- 
## 3. Demonstrasi ALUR 2: Pengguna Mencari Barang Menggunakan Teks Bebas
Pengguna mengetik apa yang mereka butuhkan. CLIP mengubahnya jadi vektor, SVM mendeteksi kategori yang dicari, dan Cosine Similarity menemukan Top-5 produk paling cocok.

In [3]:
# Contoh: Pengguna mengetik query pencarian bebas
user_query = "pisau dapur pemotong makanan"

result_alur2 = system.user_search_by_text(
    text_query=user_query,
    top_k=5
)

print("=" * 85)
print(f" 🛒 {result_alur2['workflow']}")
print(f" 🔍 QUERY PENGGUNA      : \"{result_alur2['user_query']}\"")
print(f" 🎯 KATEGORI TERDETEKSI : {result_alur2['detected_category']['category']} (Keyakinan: {result_alur2['detected_category']['confidence_pct']}%)")
print("=" * 85)
print(" 📦 TOP 5 PRODUK REKOMENDASI (COSINE SIMILARITY):")
for p in result_alur2['top_5_recommended_products']:
    print(f"  #{p['rank']} [Skor: {p['similarity_score']:.4f}] [{p['product_id']}] {p['title'][:45]:<45} ({p['category']})")
print("=" * 85)

 🛒 ALUR 2: User Semantic Text Search
 🔍 QUERY PENGGUNA      : "pisau dapur pemotong makanan"
 🎯 KATEGORI TERDETEKSI : Office Supplies (Keyakinan: 33.03%)
 📦 TOP 5 PRODUK REKOMENDASI (COSINE SIMILARITY):
  #1 [Skor: 0.2282] [prod_0510] Ralis White & Oak Kitchen Cupboard - 5 Door   (Furniture)
  #2 [Skor: 0.2263] [prod_0171] Armário de Cozinha Compacta 6 Portas 105 cm C (Furniture)
  #3 [Skor: 0.2211] [prod_0543] Organizador de Cables NANOCABLE 10.31.0301 Gr (Electronics)
  #4 [Skor: 0.2200] [prod_0792] Big Green Egg SpeediClean™ Palmyra Bristle Lo (Home & Garden)
  #5 [Skor: 0.2166] [prod_0396] Vinstativ for 73 flasker svart 45x36x200 cm s (Furniture)
